# tfm3lab — run on Kaggle (GPU)

Alternative to `run_on_colab.ipynb` for when Colab's free tier has no GPU
compute units left. Same repo, same scripts — this notebook only differs in
bootstrap plumbing (secrets, output handling), not in what actually runs.

**Before running, in this notebook's settings (right sidebar):**
1. Accelerator -> **GPU T4 x2** (or whatever GPU is offered).
2. Internet -> **On** (needed to clone from GitHub and to reach Hugging Face / TCGCSV / yfinance / FRED).
3. Add-ons -> Secrets -> add a secret named `HF_TOKEN` with a Hugging Face
   read token that has accepted the license at
   https://huggingface.co/google/timesfm-3.0-pytorch. This replaces the
   interactive `hf auth login` browser flow (which doesn't work well
   headless anyway) — `HF_TOKEN` as an env var is enough for the
   `huggingface_hub` download to authenticate.

No Drive-equivalent mount is used here: `/kaggle/working/` already persists
as this notebook's own output for the life of the session, and Kaggle
captures it automatically when you **Save Version** — no `files.download()`
hack needed, just use the Output tab (or the file browser panel) afterwards.

In [ ]:
!pip -q install uv
REPO_URL = "https://github.com/IrfEazy/timesfm3-talk"
!git clone -q $REPO_URL /kaggle/working/timesfm3-talk
%cd /kaggle/working/timesfm3-talk

In [ ]:
!uv sync --extra cuda

## Authenticate to Hugging Face via Kaggle Secrets

Requires the `HF_TOKEN` secret set up per the instructions above.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## Data root

`/kaggle/working` is this session's persistent (for the session's lifetime)
disk, unlike Colab's `/content` there's no separate ephemeral-vs-Drive split
to worry about here — but a session still ends eventually, so anything under
`/kaggle/working` only survives past that if you **Save Version** before it
does. Same env var as Colab, just pointed at Kaggle's own persistent path.

**If you already have `data/cache/*.parquet` from a previous Colab run
(e.g. sitting in your Google Drive)**, skip re-downloading entirely:
1. Download that `cache/` folder locally (drive.google.com, no Colab needed).
2. kaggle.com/datasets -> New Dataset -> upload those parquet files.
3. In this notebook's right sidebar: Add Input -> Datasets -> select it.
   It mounts read-only at `/kaggle/input/<your-dataset-slug>/`.
4. Run the "restore from an attached dataset" cell below instead of
   section 1 ("Fetch data") — then skip straight to section 2.

In [ ]:
import os

os.environ["TFM3LAB_DATA_ROOT"] = "/kaggle/working/tfm3lab-data"

### Optional: restore cache from an attached Kaggle dataset

Only run this if you attached a dataset per the instructions above. Adjust
`INPUT_DATASET_DIR` below to match your dataset's mount path (visible in
the right sidebar once attached). This copies into the writable
`TFM3LAB_DATA_ROOT` — `/kaggle/input/` itself is read-only, and
`config.RESULTS_DIR`/`FIGURES_DIR` need to be writable under the same
root, so don't point `TFM3LAB_DATA_ROOT` directly at `/kaggle/input/...`.

In [ ]:
INPUT_DATASET_DIR = "/kaggle/input/datasets/irfancela/mtgdataset"
CACHE_DIR = f"{os.environ['TFM3LAB_DATA_ROOT']}/cache"

!mkdir -p {CACHE_DIR}
!cp {INPUT_DATASET_DIR}/*.parquet {CACHE_DIR}/
!ls -la {CACHE_DIR}

## 1. Fetch data

**Skip this section entirely if you restored `data/cache/` from an
attached dataset above** — 02-05 read only from `config.CACHE_DIR`, they
don't care how it got populated.

Otherwise: the full MTG backfill (TCGCSV, ~2.5 years) is the slow part on
first run — everything downloaded is cached, so re-running this cell later
only fetches what's missing.

In [ ]:
import pathlib

_cache = pathlib.Path(os.environ["TFM3LAB_DATA_ROOT"]) / "cache"
_expected = {"mtg_prices.parquet", "market_prices.parquet", "cpi_yoy.parquet"}
if _expected.issubset({p.name for p in _cache.glob("*.parquet")}):
    print(f"Cache already populated at {_cache} (restored from a dataset?) — skipping fetch.")
else:
    !uv run scripts/00_probe_tcgcsv.py
    !uv run scripts/01_fetch_data.py

## 2-5. Run the experiments

In [ ]:
!uv run scripts/02_exp_mtg.py

In [ ]:
!uv run scripts/03_exp_shock.py

In [ ]:
!uv run scripts/04_exp_calibration.py

In [ ]:
!uv run scripts/05_exp_covariates.py

## Bring results home

`results/*.parquet` is everything the local machine needs. Copy it into
`/kaggle/working` (it already lives there via `TFM3LAB_DATA_ROOT` above, at
`/kaggle/working/tfm3lab-data/results` — this cell just confirms the path
and lists what's in it), then click **Save Version** (top right) so it
shows up under this notebook's Output tab for download, or grab it directly
from the file browser panel on the left while the session is live.

In [ ]:
RESULTS_DIR = f"{os.environ['TFM3LAB_DATA_ROOT']}/results"
print(RESULTS_DIR)
!ls -la {RESULTS_DIR}